# 6주차 ③ 재현성과 실험 관리 — 실습 6~9

**목표**: 시드를 고정해 실험을 재현하고, `state_dict` 로 가중치를 저장·복원하며,
**옵티마이저 상태까지 포함한 체크포인트로 학습을 재개**하고, TensorBoard 로 실험 5개를 한 화면에서 비교한다.

```
   월요일 : 정확도 88.4% 가 나왔다! 스크린샷을 찍어 뒀다
   목요일 : 같은 코드를 다시 돌렸다 → 86.9%
            ...어느 쪽이 진짜지? 내가 뭘 바꿨더라?
```

| 결과가 달라지는 원인 | 해결 |
|---|---|
| 가중치 **초기화**가 무작위 | `torch.manual_seed()` |
| `DataLoader` 의 **셔플** 순서 | 같은 시드 |
| 드롭아웃이 끄는 뉴런 | 같은 시드 |
| GPU 의 **비결정적 연산** | 완전히는 못 막는다 ★ |

> **핵심 메시지 ★ (출제 지점)**: 시드를 고정해도 **GPU 에서는 완전히 같지 않을 수 있습니다.**
> 병렬 연산의 덧셈 순서가 실행마다 달라질 수 있기 때문입니다.
> `torch.use_deterministic_algorithms(True)` 로 강제할 수 있지만 **느려집니다.**
> 실무에서는 **"거의 같으면 충분"** 으로 두고, 대신 **조건을 전부 기록**합니다.

> 지금은 학습이 30초라 와닿지 않습니다. **9주차 LoRA, 12주차 BERT 파인튜닝은 수십 분**입니다.
> 그때 이 습관이 없으면 실험을 통제하지 못합니다.

## 실습 6 — 시드 고정 전/후

In [ ]:
# 셀 1 — 시드를 안 고정하면
import torch, torch.nn as nn, random, numpy as np

def build():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

print("[시드 고정 안 함]")
for i in range(2):
    m = build()
    print(f"  {i+1}회차 첫 가중치 : {m[1].weight[0, :3].tolist()}")

In [ ]:
# 셀 2 — 시드 고정 함수
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # GPU 도

print("\n[시드 고정]")
for i in range(2):
    set_seed(42)
    m = build()
    print(f"  {i+1}회차 첫 가중치 : {m[1].weight[0, :3].tolist()}")

> **관찰 포인트 ★**: 위쪽은 두 번의 값이 다르고, 아래쪽은 **완전히 같습니다.**
> 이 네 줄짜리 함수가 **모든 실험 노트북의 맨 위에** 들어가야 합니다.

> 같은 폴더의 `seed_utils.py` 에 이 함수가 들어 있습니다 —
> `from seed_utils import set_seed` 로 대신할 수 있습니다.
> 체크포인트 저장/복원 도우미(`save_checkpoint`, `load_checkpoint`)도 함께 들어 있습니다.

> **함정**: 시드는 **모델을 만들기 직전**에 불러야 합니다. 노트북 맨 위에서 한 번만 부르고
> 셀을 여기저기 실행하면, 실행 순서에 따라 값이 달라집니다.

## 실습 7 — `state_dict` 저장·불러오기

```
   torch.save(model, "m.pt")               ✗ 모델 객체 통째로
        → 클래스 정의 경로에 의존한다. 다른 PC·다른 폴더 구조에서 깨진다

   torch.save(model.state_dict(), "m.pt")  ○ 가중치만  ★ 이걸 쓴다
        → 구조는 코드가 갖고 있으므로, 코드 + 가중치면 어디서든 복원된다
```

> **핵심 메시지 ★ (출제 지점)**: **`state_dict` 만 저장합니다.**
> 모델 전체 저장은 *"이 클래스가 이 경로에 있다"* 는 가정을 함께 저장하는 셈이라 깨지기 쉽습니다.

In [ ]:
# 셀 3 — 5주차 가중치를 불러온다
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
test = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
test_loader = DataLoader(test, batch_size=256)

class MLP(nn.Module):
    """★ 5주차 07_mlp_fashionmnist.ipynb 의 MLP 와 **글자 그대로 같아야** 한다.

    state_dict 는 층 이름(net.0.weight …)과 shape 으로 맞추므로,
    self.flatten / self.net 으로 나눈 구조까지 그대로 옮겨 온다.
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x): return self.net(self.flatten(x))

def accuracy(m):
    m.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (m(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t

fresh = MLP().to(device)
print(f"학습 안 한 새 모델 : {accuracy(fresh)*100:5.2f}%   ← 찍기 수준")

loaded = MLP().to(device)
loaded.load_state_dict(torch.load("models/mlp_fashion.pt", map_location=device))
print(f"5주차 가중치 로드  : {accuracy(loaded)*100:5.2f}%   ← 5주차 결과와 같아야 한다")

> **관찰 포인트 ★**: 새 모델은 10% 근처(찍기), 불러온 모델은 **5주차의 그 정확도**가 그대로 나옵니다.
> **가중치만 옮겼는데 성능이 따라옵니다.** 이게 9주차 사전학습 모델의 원리이기도 합니다.

> **함정 ★**: 구조가 조금이라도 다르면 `Missing key(s)` / `size mismatch` 오류가 납니다.
> **`state_dict` 는 층 이름과 shape 으로 맞춥니다.** 오류 메시지가 어느 층인지 알려 주니 끝까지 읽으세요.
> ⚠️ 위 클래스에서 `nn.Flatten()` 을 `self.net` 안으로 옮기기만 해도
> 키가 `net.0` → `net.1` 로 밀려 **전부 안 맞습니다.** 5주차 정의를 그대로 복사해 오세요.

> `map_location=device` 는 **GPU 에서 저장한 것을 CPU 에서 열 때** 필요합니다.
> 실습실과 개인 노트북을 오갈 때 매번 씁니다.

> **막히면**: `models/mlp_fashion.pt` 가 없으면 배포본을 쓰세요(교수 안내).

In [ ]:
# 셀 4 (참고) — state_dict 안에는 무엇이 들어 있나
sd = loaded.state_dict()
print(f"항목 {len(sd)}개\n")
for k, v in sd.items():
    print(f"  {k:20s} {tuple(v.shape)}")
print("\n→ 층 이름 + shape 의 사전일 뿐입니다. 구조 정보는 없습니다.")

## 실습 8 — 체크포인트로 학습 재개 ★★

```
   학습 3시간째에 노트북이 꺼졌다. 가중치는 저장돼 있다.
   그런데 이어서 돌리면?

     ✗ 옵티마이저가 처음 상태로 돌아간다
         Momentum 이 쌓아 둔 관성, Adam 이 쌓아 둔 통계가 전부 0
         → 손실이 한 번 튀었다가 다시 안정된다
     ✗ 지금이 몇 epoch 인지 모른다 → 스케줄러도 처음부터
```

> **핵심 메시지 ★ (출제 지점)**: 체크포인트에는 **모델 + 옵티마이저 + epoch** 를 함께 담습니다.
> 옵티마이저 상태를 빼면 재개 직후 손실이 튑니다.

In [ ]:
# 셀 5 — 5 epoch 만 돌리고 체크포인트 저장
from torch.utils.data import Subset
train_full = datasets.FashionMNIST("data", train=True, download=True, transform=tf)
train_loader = DataLoader(Subset(train_full, range(10000)), batch_size=128, shuffle=True)

set_seed(42)
model = MLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
lossfn = nn.CrossEntropyLoss()

def train_epochs(model, opt, start, end):
    for epoch in range(start, end):
        model.train(); run = n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item()*xb.size(0); n += xb.size(0)
        print(f"  epoch {epoch+1} | loss {run/n:.4f}")

train_epochs(model, opt, 0, 5)

os.makedirs("models", exist_ok=True)
torch.save({"epoch": 5,
            "model": model.state_dict(),
            "optimizer": opt.state_dict()},      # ★ 이것이 핵심
           "models/checkpoint_epoch5.pt")
print("체크포인트 저장 완료")

> ### ★ 여기서 **커널 → 다시 시작(Restart Kernel)** 하세요.
> *"학습이 끊긴 상황"* 을 실제로 만들어야 다음 셀의 의미가 삽니다.
> 재시작 후에는 **셀 1~5의 import 와 클래스 정의를 다시 실행**해야 합니다
> (학습 루프 셀은 다시 돌리지 마세요 — 그러면 재개가 아니라 처음부터가 됩니다).

In [ ]:
# 셀 6 — 커널 재시작 후, 이어서 학습
ckpt = torch.load("models/checkpoint_epoch5.pt", map_location=device)

model = MLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
model.load_state_dict(ckpt["model"])
opt.load_state_dict(ckpt["optimizer"])        # ★ 관성까지 복원
start = ckpt["epoch"]

print(f"epoch {start} 부터 재개")
train_epochs(model, opt, start, start + 3)

In [ ]:
# 셀 7 — 옵티마이저 상태를 빼면 어떻게 되나 (비교)
print("[옵티마이저 상태를 복원한 경우]")
m1 = MLP().to(device); o1 = torch.optim.SGD(m1.parameters(), lr=0.1, momentum=0.9)
m1.load_state_dict(ckpt["model"]); o1.load_state_dict(ckpt["optimizer"])
train_epochs(m1, o1, 5, 6)

print("\n[옵티마이저 상태를 빼먹은 경우]")
m2 = MLP().to(device); o2 = torch.optim.SGD(m2.parameters(), lr=0.1, momentum=0.9)
m2.load_state_dict(ckpt["model"])              # ★ 옵티마이저는 복원하지 않는다
train_epochs(m2, o2, 5, 6)

print("\n→ 아래쪽 손실이 더 높게 튀면, 그게 관성을 잃은 대가입니다")

> **관찰 포인트 ★★**: 재개 첫 epoch 의 손실이 **5 epoch 째와 자연스럽게 이어집니다.**
> 옵티마이저 상태를 빼면 손실이 한 번 튑니다.

In [ ]:
# 셀 8 — best_model.pt : 검증 성능이 좋아질 때만 덮어쓴다
val_loader = DataLoader(Subset(train_full, range(50000, 52000)), batch_size=256)

def val_loss(m):
    m.eval(); s = n = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            s += lossfn(m(xb), yb).item() * xb.size(0); n += xb.size(0)
    return s / n

set_seed(42)
model = MLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
best = float("inf")

for epoch in range(8):
    train_epochs(model, opt, epoch, epoch + 1)
    v = val_loss(model)
    mark = ""
    if v < best:                                  # ★ 2교시 조기 종료와 짝
        best = v
        torch.save(model.state_dict(), "models/best_model.pt")
        mark = "  ← best_model.pt 갱신"
    print(f"    검증 loss {v:.4f}{mark}")

print(f"\n가장 좋았던 검증 손실 : {best:.4f}")

> **핵심 메시지**: 2교시에서 본 **검증 최저점**의 가중치를 이렇게 붙잡아 둡니다.
> 마지막 epoch 의 모델이 가장 좋은 모델이 **아닐 수** 있다는 것이 2교시의 결론이었죠.

## 실습 9 — TensorBoard 로 실험 5개 비교

```
   matplotlib : 한 셀에서 그린 것만 보인다. 어제 실험은 사라졌다
   TensorBoard: 로그를 파일로 남긴다 → 언제든 다시, 여러 실험을 겹쳐서
```

In [ ]:
# 셀 9 — 5개 조건을 runs/ 에 기록
from torch.utils.tensorboard import SummaryWriter

조건 = [
    ("sgd_lr0.1",        dict(opt="sgd",  lr=0.1)),
    ("sgd_lr0.01",       dict(opt="sgd",  lr=0.01)),
    ("momentum_lr0.1",   dict(opt="mom",  lr=0.1)),
    ("adam_lr1e-3",      dict(opt="adam", lr=1e-3)),
    ("adam_lr0.1",       dict(opt="adam", lr=0.1)),      # ★ 일부러 발산시킨다
]

for name, cfg in 조건:
    set_seed(42)
    model = MLP().to(device)
    opt = {"sgd":  torch.optim.SGD(model.parameters(), lr=cfg["lr"]),
           "mom":  torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9),
           "adam": torch.optim.Adam(model.parameters(), lr=cfg["lr"])}[cfg["opt"]]

    writer = SummaryWriter(f"runs/{name}")               # ★ 실험마다 폴더 하나
    for epoch in range(5):
        model.train(); run = n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item()*xb.size(0); n += xb.size(0)
        writer.add_scalar("loss/train", run/n, epoch)     # ★ 기록
        writer.add_scalar("accuracy/test", accuracy(model), epoch)
    writer.close()                                        # ★ 안 하면 로그가 안 써진다
    print(f"{name:16s} 기록 완료")

### 보는 방법

```powershell
# 터미널에서 (venv 활성화 상태)
tensorboard --logdir runs
# → http://localhost:6006 을 브라우저에서 연다
```

In [ ]:
# 셀 10 — 방화벽이 막혔다면 노트북 안에서 연다
%load_ext tensorboard
%tensorboard --logdir runs

> **관찰 포인트 ★**: 왼쪽 목록에서 **실험을 켜고 끄면서** 곡선을 겹쳐 봅니다.
> `adam_lr0.1` 하나만 위로 치솟아 있을 겁니다 — 1교시에서 말한 *"Adam 에 0.1 을 주면 발산"* 입니다.
> **다섯 실험을 한 화면에서 비교하는 이 경험**이 실습 9의 목적입니다. **과제 제출물입니다.**

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | 페이지가 안 열린다 | 6006 포트 방화벽. `%tensorboard` 방식으로 전환 |
> | 곡선이 안 보인다 | `writer.close()` 를 안 했거나 `--logdir` 경로가 틀렸다 |
> | 실험이 하나로 합쳐진다 | `SummaryWriter` 경로를 실험마다 다르게 줬는지 확인 |
> | `No module named tensorboard` | `pip install tensorboard` |

> ⚠️ **`runs/` 를 `.gitignore` 에 추가하세요.** 로그가 쌓이면 커집니다.

---

### 중간고사 안내 (10/23 금 1교시, 50분)

| 항목 | 내용 |
|---|---|
| **범위** | **1주차 ~ 7주차** (다음 주 CNN 포함) |
| **형식** | 지필. 객관식·단답 + **코드 해석 서술** ★ |
| 배점 | 30점 (성적의 30%) |

> **오늘 배운 것 중 출제 1순위 세 개**
> - 과적합의 진단법 (훈련↓ + 검증↑)
> - `state_dict` 저장 vs 모델 전체 저장
> - `model.eval()` 을 빼면 무슨 일이 생기나

### 이 노트북 체크리스트

- [ ] 시드 고정 전후의 가중치 차이를 확인했다
- [ ] 시드를 고정해도 GPU 에서 완전히 같지 않을 수 있는 이유를 안다 ★
- [ ] `state_dict` 만 저장하는 이유를 말할 수 있다 ★
- [ ] 5주차 가중치를 새 모델에 불러와 정확도가 같은 것을 봤다
- [ ] **커널을 재시작한 뒤 체크포인트로 학습을 재개**했다 ★★
- [ ] 체크포인트에 옵티마이저를 넣는 이유를 안다
- [ ] `best_model.pt` 를 검증 최저점에 저장했다
- [ ] TensorBoard 에서 실험 5개를 겹쳐 봤다
- [ ] `runs/`, `models/` 를 `.gitignore` 에 추가했다